In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

In [3]:
# -------------------------------------------------------------------
# Setup & Getting the Data
# -------------------------------------------------------------------
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
# -------------------------------------------------------------------
# Data Preparation
# -------------------------------------------------------------------
# Categorical and numerical column identification
categorical = list(df.select_dtypes(include=['object', 'category']).columns)
numerical = list(df.select_dtypes(include=['number']).columns)

# Handle missing values
for col in categorical:
    df[col] = df[col].fillna('NA')

for col in numerical:
    df[col] = df[col].fillna(0.0)

# Ensure target column is separate
target_col = 'converted'
if target_col in numerical:
    numerical.remove(target_col)

C:\Users\saulp\AppData\Local\Temp\ipykernel_80184\4067609440.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = list(df.select_dtypes(include=['object', 'category']).columns)


In [5]:
# -------------------------------------------------------------------
# Question 1: Mode of 'industry' column
# -------------------------------------------------------------------
mode_industry = df['industry'].mode()[0]
print(f"Q1. Most frequent observation (mode) for 'industry': {mode_industry}")

Q1. Most frequent observation (mode) for 'industry': technology


In [6]:
# -------------------------------------------------------------------
# Question 2: Correlation Matrix
# -------------------------------------------------------------------
corr_matrix = df[numerical].corr()

# Check specific pairs mentioned in options
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

print("\nQ2. Correlations:")
for f1, f2 in pairs:
    val = corr_matrix.loc[f1, f2]
    print(f"    {f1} & {f2}: {val:.4f}")


Q2. Correlations:
    interaction_count & lead_score: 0.9157
    number_of_courses_viewed & lead_score: 0.7572
    number_of_courses_viewed & interaction_count: 0.7216
    annual_income & interaction_count: 0.1228


In [7]:
# -------------------------------------------------------------------
# Split the Data
# -------------------------------------------------------------------
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target_col].values
y_val = df_val[target_col].values
y_test = df_test[target_col].values

del df_train[target_col]
del df_val[target_col]
del df_test[target_col]

In [8]:
# -------------------------------------------------------------------
# Question 3: Mutual Information Scores
# -------------------------------------------------------------------
cat_features = [col for col in categorical if col != target_col]

def calculate_mi(series):
    return mutual_info_score(series, y_train)

mi_scores = df_train[cat_features].apply(calculate_mi)
mi_scores = mi_scores.round(2).sort_values(ascending=False)

print("\nQ3. Mutual Information Scores:")
print(mi_scores)


Q3. Mutual Information Scores:
lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


In [9]:
# -------------------------------------------------------------------
# Question 4: Logistic Regression & Accuracy
# -------------------------------------------------------------------
train_dicts = df_train.to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
original_acc = accuracy_score(y_val, y_pred)

print(f"\nQ4. Validation Accuracy: {round(original_acc, 2)} (exact: {original_acc:.4f})")


Q4. Validation Accuracy: 0.65 (exact: 0.6450)


In [10]:
# -------------------------------------------------------------------
# Question 5: Feature Elimination
# -------------------------------------------------------------------
all_features = list(df_train.columns)
elimination_results = {}

for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    subset_features = [f for f in all_features if f != feature]
    
    train_dict_sub = df_train[subset_features].to_dict(orient='records')
    val_dict_sub = df_val[subset_features].to_dict(orient='records')
    
    dv_sub = DictVectorizer(sparse=False)
    X_tr_sub = dv_sub.fit_transform(train_dict_sub)
    X_v_sub = dv_sub.transform(val_dict_sub)
    
    m_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m_sub.fit(X_tr_sub, y_train)
    
    acc_sub = accuracy_score(y_val, m_sub.predict(X_v_sub))
    diff = original_acc - acc_sub
    elimination_results[feature] = diff
    print(f"    Excluded '{feature}': Acc = {acc_sub:.4f}, Difference = {diff:.4f}")

    Excluded 'lead_source': Acc = 0.6420, Difference = 0.0030
    Excluded 'number_of_courses_viewed': Acc = 0.6430, Difference = 0.0020
    Excluded 'interaction_count': Acc = 0.6010, Difference = 0.0440


In [11]:
# -------------------------------------------------------------------
# Question 6: Regularization (C values)
# -------------------------------------------------------------------
c_values = [0.000001, 0.00001, 0.0001, 0.001]
print("\nQ6. Regularization evaluation:")

for C in c_values:
    model_reg = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model_reg.fit(X_train, y_train)
    
    y_pred_reg = model_reg.predict(X_val)
    score = accuracy_score(y_val, y_pred_reg)
    print(f"    C={C:<8} -> Validation Accuracy: {round(score, 3)} (exact: {score:.4f})")


Q6. Regularization evaluation:
    C=1e-06    -> Validation Accuracy: 0.598 (exact: 0.5980)
    C=1e-05    -> Validation Accuracy: 0.598 (exact: 0.5980)
    C=0.0001   -> Validation Accuracy: 0.613 (exact: 0.6130)
    C=0.001    -> Validation Accuracy: 0.645 (exact: 0.6450)
